[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leaguilar/fit_2026/blob/main/notebooks/0_fundamentos_gradient_descent.ipynb)

# 0. Gradient descent

**Taller Hands-On AI | FIT 2026**

Todo lo que veremos hoy (redes neuronales, agentes que juegan, modelos de
lenguaje) se entrena con **una sola idea**: mover unas perillas poco a poco
hasta que el error baje.

Esa idea se llama **gradient descent** y este notebook la construye
desde cero.

| Parte | Qué hacemos |
|---|---|
| 1 | Una colina en 1D. Calculamos la pendiente **a mano**. |
| 2 | Un paso de descenso. Luego muchos pasos. |
| 3 | El *learning rate*: converge, oscila o explota. |
| 4 | Dos perillas: el **mapa de niveles** y la superficie 3D. |
| 5 | Ajustamos una recta a datos reales (**TODO 1**). |

In [ ]:
# --- Setup: ejecuta esta celda primero ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation
from IPython.display import HTML, display

# Las animaciones se muestran como un reproductor dentro del notebook.
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 9

# Paleta del taller. Un color = un concepto, en TODOS los notebooks.
# Los tonos estan elegidos para que se distingan tambien con daltonismo
# rojo-verde (comprobado por script, no a ojo).
AZUL   = "#215CAF"   # el modelo / la prediccion
ROJO   = "#B7352D"   # el error y el camino que sigue el descenso
VERDE  = "#8E9C1E"   # los datos reales / la recompensa
PETROL = "#00A0C6"   # la comparacion
GRIS   = "#6F6F6F"   # ejes y contexto

# ipywidgets: en Colab a veces hay que habilitar el gestor de widgets.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass
from ipywidgets import interact, FloatSlider, Dropdown, IntSlider

print("Listo. numpy", np.__version__)

---
## 1. Aprender es bajar una colina

Nuestra colina es una parábola. La perilla se llama `w` y la altura es el
**error**. Queremos el punto más bajo.

$$ E(w) = (w - 3)^2 + 1 $$

Empezamos parados en `w = 6`, arriba a la derecha.

In [ ]:
def error(w):
    return (w - 3.0) ** 2 + 1.0

def pendiente(w):
    """Derivada de E respecto de w: dE/dw = 2*(w-3)."""
    return 2.0 * (w - 3.0)

w0 = 6.0
ws = np.linspace(-1, 7, 300)

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(ws, error(ws), color=AZUL, lw=2, label="error E(w)")
ax.plot(w0, error(w0), "o", color=ROJO, ms=10, label=f"estamos aqui: w={w0:.0f}")
ax.plot(3, error(3), "*", color=VERDE, ms=16, label="el fondo: w=3")

# La recta tangente: su inclinacion ES la pendiente.
t = np.linspace(w0 - 1.5, w0 + 1.5, 2)
ax.plot(t, error(w0) + pendiente(w0) * (t - w0), "--", color=ROJO, lw=1.5)

ax.set_xlabel("w  (la perilla)"); ax.set_ylabel("E  (el error)")
ax.set_title("La pendiente nos dice hacia donde bajar")
ax.legend(fontsize=8); ax.grid(alpha=.25); plt.show()

### A mano (2 minutos)

La derivada de $(w-3)^2 + 1$ es $2(w-3)$.

En `w = 6` vale $2 \times (6-3) = 6$.

Es **positiva**, o sea que hacia la derecha se sube. Entonces para bajar hay
que ir hacia la **izquierda**: restamos.

Esa es toda la regla:

$$ w_{\text{nuevo}} = w - \alpha \cdot \frac{dE}{dw} $$

donde $\alpha$ (*learning rate*) decide **qué tan grande** es el paso.

In [ ]:
# Comprobamos la derivada sin saber calculo: medimos la pendiente numericamente.
h = 1e-5
numerica  = (error(w0 + h) - error(w0 - h)) / (2 * h)
analitica = pendiente(w0)
print(f"pendiente numerica  = {numerica:.6f}")
print(f"pendiente analitica = {analitica:.6f}")
print("Coinciden. La computadora no necesita saber calculo para derivar.")

---
## 2. Un paso, y luego muchos

In [ ]:
alpha = 0.1
w = 6.0
print(f"paso 0: w = {w:.4f}   E = {error(w):.4f}")
w = w - alpha * pendiente(w)
print(f"paso 1: w = {w:.4f}   E = {error(w):.4f}   <- bajo el error")

In [ ]:
def descenso_1d(w_inicial, alpha, n_pasos=15):
    """Devuelve el camino completo: la lista de todas las w visitadas."""
    w = float(w_inicial)
    camino = [w]
    for _ in range(n_pasos):
        w = w - alpha * pendiente(w)
        camino.append(w)
    return np.array(camino)

camino = descenso_1d(6.0, alpha=0.1, n_pasos=15)

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(ws, error(ws), color=AZUL, lw=2)
ax.plot(camino, error(camino), "o-", color=ROJO, ms=5, lw=1, label="camino")
ax.plot(3, error(3), "*", color=VERDE, ms=16, label="el fondo")
ax.set_xlabel("w"); ax.set_ylabel("E"); ax.legend(fontsize=8); ax.grid(alpha=.25)
ax.set_title(f"15 pasos con alpha=0.1: de w={camino[0]:.1f} a w={camino[-1]:.3f}")
plt.show()

In [ ]:
# La misma bajada, animada. Usa el boton de play.
camino = descenso_1d(6.0, alpha=0.1, n_pasos=40)

fig, ax = plt.subplots(figsize=(6, 3.4), dpi=80)
ax.plot(ws, error(ws), color=AZUL, lw=2)
ax.plot(3, error(3), "*", color=VERDE, ms=16)
rastro, = ax.plot([], [], "-", color=ROJO, lw=1, alpha=.6)
pelota, = ax.plot([], [], "o", color=ROJO, ms=12)
ax.set_xlabel("w"); ax.set_ylabel("E"); ax.grid(alpha=.25)
titulo = ax.set_title("")

def animar(k):
    rastro.set_data(camino[:k + 1], error(camino[:k + 1]))
    pelota.set_data([camino[k]], [error(camino[k])])
    titulo.set_text(f"paso {k:2d}   w = {camino[k]:.3f}   E = {error(camino[k]):.3f}")
    return rastro, pelota, titulo

ani = matplotlib.animation.FuncAnimation(fig, animar, frames=len(camino), interval=120)
plt.close(fig)
ani

---
## 3. El *learning rate* lo decide todo

Mueve el deslizador. Hay tres regímenes y vale la pena verlos:

* `alpha` muy chico: baja, pero tarda una eternidad.
* `alpha` justo: llega al fondo en pocos pasos.
* `alpha` grande: se pasa de largo y **rebota**.
* `alpha` > 1: **explota**, el error crece sin parar.

In [ ]:
def dibujar_lr(alpha=0.1, n_pasos=15):
    camino = descenso_1d(6.0, alpha, n_pasos)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2))

    a1.plot(ws, error(ws), color=AZUL, lw=2)
    a1.plot(camino, error(camino), "o-", color=ROJO, ms=5, lw=1)
    a1.plot(3, error(3), "*", color=VERDE, ms=16)
    a1.set_xlim(-1, 7); a1.set_ylim(0, 30)
    a1.set_xlabel("w"); a1.set_ylabel("E"); a1.grid(alpha=.25)
    a1.set_title(f"alpha = {alpha:.3f}")

    a2.plot(error(camino), "o-", color=ROJO, ms=4)
    a2.set_yscale("log"); a2.set_xlabel("paso"); a2.set_ylabel("E (escala log)")
    a2.grid(alpha=.25); a2.set_title(f"E final = {error(camino[-1]):.4g}")
    plt.tight_layout(); plt.show()

interact(dibujar_lr,
         alpha=FloatSlider(value=0.1, min=0.01, max=1.05, step=0.01,
                           description="alpha", continuous_update=False),
         n_pasos=IntSlider(value=15, min=2, max=40, step=1, description="pasos"))

In [ ]:
# Los cuatro regimenes, uno al lado del otro. Si el deslizador no te funciona,
# esta celda cuenta la misma historia.
fig, axes = plt.subplots(1, 4, figsize=(12, 2.8), sharey=True)
regimenes = [(0.01, "muy lento"), (0.1, "justo"), (0.9, "rebota"), (1.02, "explota")]
for ax, (a, etiqueta) in zip(axes, regimenes):
    c = descenso_1d(6.0, a, 15)
    ax.plot(ws, error(ws), color=AZUL, lw=1.5)
    ax.plot(c, error(c), "o-", color=ROJO, ms=4, lw=1)
    ax.plot(3, error(3), "*", color=VERDE, ms=13)
    ax.set_xlim(-1, 7); ax.set_ylim(0, 30); ax.grid(alpha=.25)
    ax.set_title(f"alpha={a}  ({etiqueta})\nE final={error(c[-1]):.3g}", fontsize=9)
    ax.set_xlabel("w")
axes[0].set_ylabel("E")
plt.tight_layout(); plt.show()

---
## 4. Dos perillas: el mapa de niveles

Con una perilla el error es una curva. Con **dos** perillas (`w` y `b`) el
error es una **superficie**, y la dibujamos de dos maneras:

* **Mapa de niveles** (izquierda): visto desde arriba, como un mapa de
  montaña. Cada anillo es una altura. El fondo es el centro.
* **Superficie 3D** (centro): la misma cosa, en perspectiva.

El gradient descent siempre camina **perpendicular a los anillos**,
hacia adentro. Eso es todo lo que hace.

In [ ]:
# Tres paisajes distintos. Cada uno enseña algo.
PAISAJES = {
    # tazon simetrico: el caso facil, el camino es casi recto
    "tazon":  (lambda w, b: 0.5 * (w - 3) ** 2 + 0.5 * (b + 2) ** 2,
               lambda w, b: ((w - 3), (b + 2))),
    # valle estirado: baja mucho mas rapido en b que en w -> zigzag
    "valle":  (lambda w, b: 0.5 * (w - 3) ** 2 + 8.0 * (b + 2) ** 2,
               lambda w, b: ((w - 3), 16.0 * (b + 2))),
    # silla de montar: no hay fondo, se escapa por los lados
    "silla":  (lambda w, b: 0.15 * (w ** 2 - b ** 2),
               lambda w, b: (0.3 * w, -0.3 * b)),
}

def descenso_2d(paisaje, w0, b0, alpha, n_pasos=60):
    _, grad = PAISAJES[paisaje]
    w, b = float(w0), float(b0)
    camino = [(w, b)]
    for _ in range(n_pasos):
        dw, db = grad(w, b)
        w, b = w - alpha * dw, b - alpha * db
        if not np.isfinite(w) or not np.isfinite(b) or abs(w) > 1e4:
            break
        camino.append((w, b))
    return np.array(camino)

def tres_vistas(paisaje="tazon", alpha=0.10, w0=-1.5, b0=2.5, n_pasos=60,
                con_3d=True):
    f, _ = PAISAJES[paisaje]
    camino = descenso_2d(paisaje, w0, b0, alpha, n_pasos)
    cw, cb = camino[:, 0], camino[:, 1]

    # Ventana cuadrada centrada en el camino, con margen: asi el paisaje se ve
    # completo y la superficie 3D parece un tazon y no una esquina.
    cx, cy = (cw.min() + cw.max()) / 2, (cb.min() + cb.max()) / 2
    r = min(max(np.ptp(cw), np.ptp(cb)) * 0.9 + 1.5, 22)
    W, B = np.meshgrid(np.linspace(cx - r, cx + r, 180),
                       np.linspace(cy - r, cy + r, 180))
    Z = f(W, B)
    # Anillos mas juntos cerca del fondo, para que se vea la estructura.
    niveles = Z.min() + (Z.max() - Z.min()) * np.linspace(0, 1, 22) ** 2

    n_paneles = 3 if con_3d else 2
    fig = plt.figure(figsize=(12 if con_3d else 8.6, 3.6))

    # (1) el mapa de niveles: claro = poco error, oscuro = mucho error
    a1 = fig.add_subplot(1, n_paneles, 1)
    a1.contourf(W, B, Z, levels=niveles, cmap="Blues", alpha=.9)
    a1.contour(W, B, Z, levels=niveles, colors=GRIS, linewidths=.4, alpha=.5)
    a1.plot(cw, cb, "o-", color=ROJO, ms=3, lw=1.4)
    a1.plot(cw[0], cb[0], "o", color=ROJO, ms=10, mec="white", mew=.8)
    a1.plot(cw[-1], cb[-1], "*", color=VERDE, ms=18, mec="white", mew=.8)
    a1.set_xlabel("w"); a1.set_ylabel("b")
    a1.set_title("mapa de niveles (visto desde arriba)")

    # (2) la misma superficie en 3D. Recortamos las paredes mas altas: si no,
    #     el tazon se aplasta y no se ve nada.
    if con_3d:
        a2 = fig.add_subplot(1, 3, 2, projection="3d")
        techo = np.percentile(Z, 90)
        Zc = np.minimum(Z, techo)
        piso = Z.min() - 0.18 * (techo - Z.min())
        a2.plot_surface(W, B, Zc, cmap="Blues", alpha=.9, linewidth=0,
                        rstride=4, cstride=4, antialiased=False)
        # Las mismas curvas de nivel del panel 1, proyectadas en el piso.
        a2.contour(W, B, Zc, levels=14, zdir="z", offset=piso,
                   colors=GRIS, linewidths=.5, alpha=.6)
        salto = (techo - Z.min()) * 0.04    # levanta el camino sobre la superficie
        a2.plot(cw, cb, np.minimum(f(cw, cb), techo) + salto,
                "-", color=ROJO, lw=2.5, zorder=10)
        a2.plot(cw, cb, np.full_like(cw, piso), "-", color=ROJO, lw=1, alpha=.5)
        a2.set_zlim(piso, techo)
        a2.set_xlabel("w"); a2.set_ylabel("b"); a2.set_zlabel("E")
        a2.set_title("la misma superficie, en 3D")
        a2.view_init(elev=30, azim=-62)

    # (3) el error contra el numero de paso
    a3 = fig.add_subplot(1, n_paneles, n_paneles)
    a3.plot(f(cw, cb), "o-", color=ROJO, ms=3)
    a3.set_xlabel("paso"); a3.set_ylabel("E"); a3.grid(alpha=.25)
    a3.set_title(f"{len(camino)-1} pasos, E final = {f(cw[-1], cb[-1]):.4g}")

    plt.tight_layout(); plt.show()

tres_vistas()

In [ ]:
# Juega: cambia el paisaje, el punto de partida y el tamano del paso.
#   - "valle" con alpha grande -> el camino hace ZIGZAG entre las paredes.
#   - "silla" -> no hay fondo, el camino se escapa. No siempre hay solucion.
def explorar(paisaje="tazon", alpha=0.10, w0=-1.5, b0=2.5, n_pasos=60):
    # Aqui dejamos solo el mapa de niveles y la curva de error: son los dos
    # paneles donde se lee lo que hace cada deslizador.
    tres_vistas(paisaje, alpha, w0, b0, n_pasos, con_3d=False)

interact(explorar,
         paisaje=Dropdown(options=list(PAISAJES), value="tazon", description="paisaje"),
         alpha=FloatSlider(value=0.10, min=0.01, max=0.14, step=0.005,
                           description="alpha", readout_format=".3f",
                           continuous_update=False),
         w0=FloatSlider(value=-1.5, min=-6, max=8, step=.5, description="w inicial",
                        continuous_update=False),
         b0=FloatSlider(value=2.5, min=-7, max=5, step=.5, description="b inicial",
                        continuous_update=False),
         n_pasos=IntSlider(value=60, min=5, max=120, step=5, description="pasos"))

In [ ]:
# El camino sobre el mapa de niveles, animado paso a paso.
paisaje = "valle"          # prueba tambien "tazon"
f, _ = PAISAJES[paisaje]
camino = descenso_2d(paisaje, w0=-1.5, b0=2.5, alpha=0.11, n_pasos=60)
W, B = np.meshgrid(np.linspace(-6, 8, 200), np.linspace(-7, 5, 200))

fig, ax = plt.subplots(figsize=(5.2, 3.8), dpi=80)
ax.contourf(W, B, f(W, B), levels=28, cmap="Blues", alpha=.9)
ax.contour(W, B, f(W, B), levels=28, colors=GRIS, linewidths=.4, alpha=.5)
rastro, = ax.plot([], [], "-", color=ROJO, lw=1.4)
punto,  = ax.plot([], [], "o", color=ROJO, ms=9)
ax.set_xlabel("w"); ax.set_ylabel("b")
titulo = ax.set_title("")

def animar2(k):
    rastro.set_data(camino[:k + 1, 0], camino[:k + 1, 1])
    punto.set_data([camino[k, 0]], [camino[k, 1]])
    titulo.set_text(f"paisaje '{paisaje}'   paso {k:2d}   E = {f(*camino[k]):.4f}")
    return rastro, punto, titulo

ani2 = matplotlib.animation.FuncAnimation(fig, animar2, frames=len(camino), interval=110)
plt.close(fig)
ani2

---
## 5. Ahora con datos reales: ajustar una recta

Tenemos tres puntos y queremos la recta $y = w\,x + b$ que mejor pasa por
ellos. El error es la distancia vertical promedio, al cuadrado:

$$ E(w,b) = \frac{1}{2n}\sum_i \left(w\,x_i + b - y_i\right)^2 $$

Nadie nos dice cuáles son la `w` y la `b` buenas. Las **encuentra** el
gradient descent, exactamente igual que en la parábola.

In [ ]:
xs = np.array([4.0, 3.0, 2.0])
ys = np.array([3.0, 2.0, 1.0])

def error_y_gradientes(w, b, xs, ys):
    pred = w * xs + b
    err  = pred - ys
    E    = 0.5 * np.mean(err ** 2)
    dw   = np.mean(err * xs)        # dE/dw
    db   = np.mean(err)             # dE/db
    return E, dw, db

print("En w=0, b=0:", error_y_gradientes(0.0, 0.0, xs, ys))

### TODO 1

La función de abajo **no actualiza nada**: devuelve la `w` y la `b` sin
tocarlas. Arréglala.

Pista: hay que restar el gradiente multiplicado por el *learning rate*.

In [ ]:
# === TODO 1 =========================================================
LR = 0.1

def paso(w, b, dw, db):
    w_nuevo = w        # <-- TODO: cambia esto
    b_nuevo = b        # <-- TODO: y esto
    return w_nuevo, b_nuevo
# ====================================================================

def entrenar(paso_fn, n=240):
    w, b = 0.0, 0.0
    camino = [(w, b)]
    for _ in range(n):
        _, dw, db = error_y_gradientes(w, b, xs, ys)
        w, b = paso_fn(w, b, dw, db)
        camino.append((w, b))
    return np.array(camino)

c = entrenar(paso)
E_fin = error_y_gradientes(c[-1, 0], c[-1, 1], xs, ys)[0]
print(f"w={c[-1,0]:.4f}  b={c[-1,1]:.4f}  E={E_fin:.4f}")
print("Si E sigue valiendo 2.33, la funcion todavia no hace nada.")

### Solución

Ejecuta esta celda **después** de intentarlo.

In [ ]:
def paso(w, b, dw, db):
    return w - LR * dw, b - LR * db

camino_rec = entrenar(paso)
w_fin, b_fin = camino_rec[-1]
print(f"w={w_fin:.4f}  b={b_fin:.4f}  "
      f"E={error_y_gradientes(w_fin, b_fin, xs, ys)[0]:.6f}")
print("La recta verdadera es y = 1*x - 1.")

In [ ]:
# Las dos imagenes de lo mismo, animadas a la vez:
#   izquierda  -> la recta acomodandose sobre los datos
#   derecha    -> el punto (w,b) bajando por el mapa de niveles del error
sub = camino_rec[::4]                      # 1 de cada 4 pasos -> 61 cuadros
Wg, Bg = np.meshgrid(np.linspace(-0.5, 2.0, 200), np.linspace(-2.0, 0.6, 200))
Eg = 0.5 * np.mean([(Wg * x + Bg - y) ** 2 for x, y in zip(xs, ys)], axis=0)
xline = np.linspace(1.5, 4.5, 2)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.5, 3.6), dpi=72)
a1.plot(xs, ys, "o", color=VERDE, ms=10, label="datos")
recta, = a1.plot([], [], color=AZUL, lw=2, label="nuestra recta")
a1.set_xlim(1.5, 4.5); a1.set_ylim(-1.5, 3.8)
a1.set_xlabel("x"); a1.set_ylabel("y"); a1.legend(fontsize=8); a1.grid(alpha=.25)

a2.contourf(Wg, Bg, Eg, levels=30, cmap="Blues", alpha=.9)
a2.contour(Wg, Bg, Eg, levels=30, colors=GRIS, linewidths=.4, alpha=.5)
a2.plot(1.0, -1.0, "*", color=VERDE, ms=17, mec="white", mew=.8)
rastro3, = a2.plot([], [], "-", color=ROJO, lw=1.4)
punto3,  = a2.plot([], [], "o", color=ROJO, ms=9)
a2.set_xlabel("w"); a2.set_ylabel("b"); a2.set_title("mapa de niveles del error")
t1 = a1.set_title("")

def animar3(k):
    w, b = sub[k]
    recta.set_data(xline, w * xline + b)
    rastro3.set_data(sub[:k + 1, 0], sub[:k + 1, 1])
    punto3.set_data([w], [b])
    t1.set_text(f"paso {k*4:3d}   w={w:.3f}  b={b:.3f}")
    return recta, rastro3, punto3, t1

ani3 = matplotlib.animation.FuncAnimation(fig, animar3, frames=len(sub), interval=90)
plt.close(fig)
ani3

---
## Resumen

1. El **gradiente** dice hacia dónde sube el error.
2. Restarlo hace bajar el error: $w \leftarrow w - \alpha \nabla E$.
3. El *learning rate* $\alpha$ decide si converge, rebota o explota.
4. Con dos o más perillas es exactamente lo mismo, solo que el paisaje es
   una superficie y el camino se ve en el mapa de niveles.

Una red neuronal moderna hace **esto mismo**, con miles de millones de
perillas en vez de dos. Lo único que falta es saber calcular el gradiente
cuando hay muchas capas encadenadas.

Eso se llama **backpropagation** y es el siguiente notebook.